# Tarea 3 — PyTorch

**Nombre del integrante:** Diego Mendez

**Cédula del integrante:** 30830746

> Enunciado, rúbrica y nombres de los archivos `.pth` a entregar: ver `README.md` de esta carpeta.
> Antes del commit: *Restart & Run All* debe terminar sin errores y quedar en el menú.

## 1. Imports y constantes

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(30830746)

INPUT_SIZE = 784
DATA_DIR = "data"
BATCH_SIZE = 128

DEFAULT_HIDDEN_LAYERS = 2
DEFAULT_NEURONS = 512

CLASS_NAMES = {
    "original": ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
                 "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"],
    "simplified": ["Top", "Bottom", "Footwear", "Bag"],
}

# Fashion MNIST label -> simplified label (0 = Top, 1 = Bottom, 2 = Footwear, 3 = Bag)
SIMPLIFIED_MAP = torch.tensor([0, 1, 0, 1, 0, 2, 0, 2, 3, 2])

# Fields of the network dictionary that are written to the .pth file
HYPERPARAMETER_KEYS = ["network_type", "classification", "hidden_layers", "hidden_sizes",
                       "n_outputs", "epochs", "batch_norm", "dropout", "optimizer",
                       "learning_rate", "weight_decay", "early_stopping"]

## 2. Datos (Fashion MNIST original y clases simplificadas)

In [ ]:
DATA_CACHE = {}


def load_data(classification):
    """Returns (train_loader, test_loader). Pixels stay in [0, 1] (ToTensor only)."""
    if classification in DATA_CACHE:
        return DATA_CACHE[classification]

    train_set = datasets.FashionMNIST(DATA_DIR, train=True, download=True,
                                      transform=transforms.ToTensor())
    test_set = datasets.FashionMNIST(DATA_DIR, train=False, download=True,
                                     transform=transforms.ToTensor())

    if classification == "simplified":
        train_set.targets = SIMPLIFIED_MAP[train_set.targets]
        test_set.targets = SIMPLIFIED_MAP[test_set.targets]

    loaders = (DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True),
               DataLoader(test_set, batch_size=1000))
    DATA_CACHE[classification] = loaders
    return loaders

## 3. Modelos (rectangular y convolucional)

In [ ]:
def hidden_sizes(network_type, hidden_layers, neurons, n_outputs):
    """Neurons of each hidden layer."""
    if network_type == "rectangular":
        return [neurons] * hidden_layers

    # "convolutional": every layer loses the same amount of neurons (linear interpolation)
    sizes = []
    for i in range(1, hidden_layers + 1):
        sizes.append(round(INPUT_SIZE + (n_outputs - INPUT_SIZE) * i / (hidden_layers + 1)))
    return sizes


def build_model(net):
    """nn.Sequential: Flatten, then [Linear, BatchNorm, ReLU, Dropout] per hidden layer, then Linear."""
    layers = [nn.Flatten()]
    previous = INPUT_SIZE
    for size in net["hidden_sizes"]:
        layers.append(nn.Linear(previous, size))
        if net["batch_norm"]:
            layers.append(nn.BatchNorm1d(size))
        layers.append(nn.ReLU())
        if net["dropout"] > 0:
            layers.append(nn.Dropout(net["dropout"]))
        previous = size
    layers.append(nn.Linear(previous, net["n_outputs"]))
    return nn.Sequential(*layers)


def create_network(classification, network_type, hidden_layers, neurons,
                   batch_norm=False, dropout=0.0):
    n_outputs = len(CLASS_NAMES[classification])
    net = {
        "network_type": network_type,
        "classification": classification,
        "hidden_layers": hidden_layers,
        "hidden_sizes": hidden_sizes(network_type, hidden_layers, neurons, n_outputs),
        "n_outputs": n_outputs,
        "epochs": 0,
        # optimizations
        "batch_norm": batch_norm,
        "dropout": dropout,
        "optimizer": "adam",
        "learning_rate": 0.001,
        "weight_decay": 0.0,
        "early_stopping": 0,    # patience in epochs, 0 = disabled
    }
    net["model"] = build_model(net)
    return net

## 4. Entrenar, probar, guardar y cargar

In [ ]:
def make_optimizer(net):
    parameters = net["model"].parameters()
    if net["optimizer"] == "sgd":
        return torch.optim.SGD(parameters, lr=net["learning_rate"], momentum=0.9,
                               weight_decay=net["weight_decay"])
    return torch.optim.Adam(parameters, lr=net["learning_rate"],
                            weight_decay=net["weight_decay"])


def train_network(net, epochs):
    """Trains for `epochs` epochs (fewer if early stopping triggers). Returns the loss of each epoch."""
    train_loader, _ = load_data(net["classification"])
    model = net["model"]
    optimizer = make_optimizer(net)
    loss_function = nn.CrossEntropyLoss()

    losses = []
    best_loss = float("inf")
    epochs_without_improvement = 0
    model.train()

    for _ in range(epochs):
        total = 0.0
        for images, labels in train_loader:
            optimizer.zero_grad()
            loss = loss_function(model(images), labels)
            loss.backward()
            optimizer.step()
            total += loss.item() * len(labels)

        losses.append(total / len(train_loader.dataset))
        net["epochs"] += 1
        print(f"  Epoca {net['epochs']}: perdida = {losses[-1]:.5f}")

        if net["early_stopping"] > 0:
            if losses[-1] < best_loss:
                best_loss = losses[-1]
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
                if epochs_without_improvement >= net["early_stopping"]:
                    print("  Early stopping: la perdida dejo de mejorar.")
                    break

    return losses


def evaluate_network(net):
    """Confusion matrix on the test set. Rows = real class, columns = predicted class."""
    _, test_loader = load_data(net["classification"])
    n = net["n_outputs"]
    matrix = [[0] * n for _ in range(n)]

    net["model"].eval()
    with torch.no_grad():
        for images, labels in test_loader:
            predicted = net["model"](images).argmax(dim=1)
            for real, guess in zip(labels.tolist(), predicted.tolist()):
                matrix[real][guess] += 1
    return matrix


def compute_metrics(matrix):
    """Returns (accuracy, precision per class, recall per class)."""
    n = len(matrix)
    total = sum(sum(row) for row in matrix)
    correct = sum(matrix[i][i] for i in range(n))

    precision = []
    recall = []
    for c in range(n):
        predicted_as_c = sum(matrix[i][c] for i in range(n))    # column
        real_c = sum(matrix[c])                                 # row
        precision.append(matrix[c][c] / predicted_as_c if predicted_as_c > 0 else 0.0)
        recall.append(matrix[c][c] / real_c if real_c > 0 else 0.0)

    return correct / total, precision, recall


def print_metrics(matrix, names):
    accuracy, precision, recall = compute_metrics(matrix)
    print(f"\nAccuracy: {100 * accuracy:.2f}%")
    print(f"{'Clase':<14}{'Precision':>10}{'Recall':>10}")
    for name, p, r in zip(names, precision, recall):
        print(f"{name:<14}{100 * p:>9.2f}%{100 * r:>9.2f}%")
    print(f"{'Promedio':<14}{100 * sum(precision) / len(precision):>9.2f}%"
          f"{100 * sum(recall) / len(recall):>9.2f}%")


def plot_loss(losses, first_epoch):
    epochs = list(range(first_epoch, first_epoch + len(losses)))
    plt.figure(figsize=(7, 4))
    plt.plot(epochs, losses, color="#3b6fb6", linewidth=1.8, marker="o")
    plt.xlabel("Epoca")
    plt.ylabel("Perdida")
    plt.title("Perdida por epoca")
    plt.grid(True, alpha=0.25)
    plt.show()


def plot_confusion_matrix(matrix, names):
    n = len(matrix)
    maximum = max(max(row) for row in matrix)
    plt.figure(figsize=(7, 6))
    plt.matshow(matrix, fignum=0, cmap="Blues")
    for i in range(n):
        for j in range(n):
            color = "white" if maximum > 0 and matrix[i][j] > maximum / 2 else "black"
            plt.text(j, i, str(matrix[i][j]), ha="center", va="center", color=color)
    plt.xticks(range(n), names, rotation=45, ha="right")
    plt.yticks(range(n), names)
    plt.gca().xaxis.set_ticks_position("bottom")
    plt.xlabel("Clase predicha")
    plt.ylabel("Clase real")
    plt.title("Matriz de confusion")
    plt.tight_layout()
    plt.show()


def save_network(net, path):
    hyperparameters = {key: net[key] for key in HYPERPARAMETER_KEYS}
    torch.save({"state_dict": net["model"].state_dict(), "hyperparameters": hyperparameters}, path)


def load_network(path):
    """Returns the network dictionary, or None if the file does not exist."""
    try:
        checkpoint = torch.load(path, weights_only=True)
    except FileNotFoundError:
        return None
    net = dict(checkpoint["hyperparameters"])
    net["model"] = build_model(net)
    net["model"].load_state_dict(checkpoint["state_dict"])
    return net

## 5. Menú

La última celda debe dejar el menú corriendo (crear/cargar, entrenar, probar, guardar, salir).

In [ ]:
def ask_integer(message, minimum, default=None):
    while True:
        text = input(message).strip()
        if text == "" and default is not None:
            return default
        try:
            value = int(text)
        except ValueError:
            print("  -> Ingresa un numero entero.")
            continue
        if value < minimum:
            print(f"  -> Debe ser al menos {minimum}.")
            continue
        return value


def ask_decimal(message, minimum, maximum, default):
    while True:
        text = input(message).strip()
        if text == "":
            return default
        try:
            value = float(text)
        except ValueError:
            print("  -> Ingresa un numero.")
            continue
        if not minimum <= value <= maximum:
            print(f"  -> Debe estar entre {minimum} y {maximum}.")
            continue
        return value


def ask_choice(message, options, default):
    """options: {"1": value, ...}. Enter selects the default key."""
    while True:
        text = input(message).strip() or default
        if text in options:
            return options[text]
        print("  -> Opcion invalida.")


def ask_yes_no(message):
    return input(message).strip().lower() == "s"


def print_hyperparameters(net):
    print("\n--- Hiperparametros ---")
    print(f"  Clasificacion:        {net['classification']} ({net['n_outputs']} clases)")
    print(f"  Tipo de red:          {net['network_type']}")
    print(f"  Capas ocultas:        {net['hidden_layers']} {net['hidden_sizes']}")
    print(f"  Epocas entrenadas:    {net['epochs']}")
    print(f"  Batch normalization:  {'si' if net['batch_norm'] else 'no'}")
    print(f"  Dropout:              {net['dropout']}")
    print(f"  Optimizador:          {net['optimizer']}")
    print(f"  Learning rate:        {net['learning_rate']}")
    print(f"  Weight decay:         {net['weight_decay']}")
    if net["early_stopping"]:
        print(f"  Early stopping:       paciencia {net['early_stopping']}")
    else:
        print("  Early stopping:       no")


def create_network_interactive():
    print("\n--- Nueva red (Enter = valor por defecto) ---")
    classification = ask_choice("Clasificacion: 1) original (10 clases)  2) simplificada (4 clases) [1]: ",
                                {"1": "original", "2": "simplified"}, "1")
    hidden_layers = ask_integer(f"Numero de capas ocultas [{DEFAULT_HIDDEN_LAYERS}]: ", 0,
                                DEFAULT_HIDDEN_LAYERS)
    network_type = ask_choice("Tipo de red: 1) rectangular  2) convolucional [1]: ",
                              {"1": "rectangular", "2": "convolutional"}, "1")
    neurons = DEFAULT_NEURONS
    if network_type == "rectangular" and hidden_layers > 0:
        neurons = ask_integer(f"Neuronas por capa [{DEFAULT_NEURONS}]: ", 1, DEFAULT_NEURONS)

    # these two change the architecture, so they can only be chosen here
    batch_norm = ask_yes_no("Usar batch normalization? (s/n) [n]: ")
    dropout = ask_decimal("Dropout, 0 = sin dropout [0]: ", 0.0, 0.9, 0.0)

    return create_network(classification, network_type, hidden_layers, neurons,
                          batch_norm, dropout)


def load_network_interactive():
    path = input("Archivo de la red (.pth): ").strip()
    try:
        net = load_network(path)
    except Exception as error:
        print(f"No se pudo cargar la red: {error}")
        return None
    if net is None:
        print("No se encontro el archivo")
    return net


def optimizations_menu(net):
    """Optimizations that do not change the architecture."""
    while True:
        print_hyperparameters(net)
        print("\n--- Optimizaciones ---")
        print("1) Optimizador (adam / sgd)")
        print("2) Learning rate")
        print("3) Weight decay")
        print("4) Early stopping")
        print("5) Volver")
        option = input("Elige una opcion (1-5): ").strip()

        if option == "1":
            net["optimizer"] = ask_choice("Optimizador: 1) adam  2) sgd [1]: ",
                                          {"1": "adam", "2": "sgd"}, "1")
        elif option == "2":
            net["learning_rate"] = ask_decimal("Learning rate (0.00001 a 1) [0.001]: ", 1e-5, 1.0, 0.001)
        elif option == "3":
            net["weight_decay"] = ask_decimal("Weight decay (0 a 0.1) [0]: ", 0.0, 0.1, 0.0)
        elif option == "4":
            net["early_stopping"] = ask_integer("Paciencia en epocas (0 = desactivado) [0]: ", 0, 0)
        elif option == "5":
            return
        else:
            print("  -> Opcion invalida.")


def option_train(net):
    epochs = ask_integer("Numero de epocas: ", 1)
    first_epoch = net["epochs"] + 1

    print("\nEntrenando...")
    losses = train_network(net, epochs)

    print(f"\nPerdida final: {losses[-1]:.5f}")
    plot_loss(losses, first_epoch)


def option_test(net):
    names = CLASS_NAMES[net["classification"]]
    matrix = evaluate_network(net)
    print_metrics(matrix, names)
    print_hyperparameters(net)
    plot_confusion_matrix(matrix, names)


def option_save(net):
    suggestion = f"pytorch_fashion_3_{net['classification']}_{net['network_type']}.pth"
    path = input(f"Nombre del archivo [{suggestion}]: ").strip() or suggestion
    try:
        save_network(net, path)
    except OSError as error:
        print(f"No se pudo guardar: {error}")
        return False
    print(f"Red guardada en {path}")
    return True

In [ ]:
def main():
    print("=== Clasificador Fashion MNIST (PyTorch) ===")

    net = None
    while net is None:
        print("\n1) Crear una nueva red")
        print("2) Cargarla de un archivo")
        option = input("Elige una opcion (1/2): ").strip()
        if option == "1":
            net = create_network_interactive()
        elif option == "2":
            net = load_network_interactive()
        else:
            print("  -> Opcion invalida.")
    print_hyperparameters(net)

    unsaved = False
    while True:
        print("\n--- Menu ---")
        print("1) Entrenar")
        print("2) Probar con el conjunto de prueba")
        print("3) Guardar la red")
        print("4) Optimizaciones")
        print("5) Salir")
        option = input("Elige una opcion (1-5): ").strip()

        if option == "1":
            option_train(net)
            unsaved = True
        elif option == "2":
            option_test(net)
        elif option == "3":
            if option_save(net):
                unsaved = False
        elif option == "4":
            optimizations_menu(net)
        elif option == "5":
            if unsaved:
                answer = input("Hay entrenamiento sin guardar. ¿Salir de todos modos? (s/n): ")
                if answer.strip().lower() != "s":
                    continue
            break
        else:
            print("  -> Opcion invalida.")

    print("Fin")


main()